<a href="https://colab.research.google.com/github/ingridevv/data-engineering-zoomcamp/blob/main/week_6_batch/01_pyspark_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

In [ ]:
# Initialize spark session
spark = SparkSession.builder \
    .appName('spark_intro') \
    .getOrCreate()

## Start Tunnel to Access SparkUI with ngrok

In [ ]:
from pyngrok import ngrok, conf
import getpass

conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

··········


 * ngrok tunnel "https://senaida-interlobular-franklyn.ngrok-free.dev" -> "http://127.0.0.1:4040"


## Download Yellow Taxi Data, Read it with Spark

In [ ]:
# Download file directly
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-05 10:42:59--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-05T11%3A17%3A50Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-05T10%3A17%3A46Z&ske=2026-03-05T11%3A17%3A50Z&sks=b&skv=2018-11-09&sig=2A%2BY3nFXaToPXppndxrWp1bIkZ59kN4L2bZKu7PCok0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjcxMDk4MCwibmJmIjoxNzcyNzA3MzgwLCJwYXRoIj

In [ ]:
# Read the file (Spark unzip automatically)
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("fhvhv_tripdata_2021-01.csv.gz")

df.count()

11908468

In [ ]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', IntegerType(), True)])

In [ ]:
df.show(5)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
+-----------------+--------------------+-------------------+-------------------+

In [ ]:
# Read the first 101 rows with pandas (zcat unzip first)
!zcat fhvhv_tripdata_2021-01.csv | head -n 101 > head.csv

In [ ]:
!wc -l head.csv

101 head.csv


In [ ]:
import pandas as pd

df_pandas = pd.read_csv('head.csv')

In [ ]:
df_pandas.dtypes

,0
hvfhs_license_num,object
dispatching_base_num,object
pickup_datetime,object
dropoff_datetime,object
PULocationID,int64
DOLocationID,int64
SR_Flag,float64


In [ ]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [ ]:
from pyspark.sql import types

In [ ]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)]
)

In [ ]:
# Use our defined schema
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("fhvhv_tripdata_2021-01.csv.gz")

In [ ]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

### Save as parquet files
*Note: Remember that Spark is a cluster with multiple executors that can handle one file at a time*
We should cut this one large file and split into partitions -> df.repartition()

In [ ]:
df = df.repartition(24)

In [ ]:
df.write.parquet('fhvhv/2021/01/')

In [ ]:
!ls -lh fhvhv/2021/01/

total 226M
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00000-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00001-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00002-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00003-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00004-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00005-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00006-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00007-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r-- 1 root root 9.4M Mar  5 11:06 part-00008-221d9fda-3814-4b0b-90f4-f052e32aa8c1-c000.snappy.parquet
-rw-r--r

## Disconnect Ngrok public endpoint

In [ ]:
# Optionally put the tunnel down
# ngrok.disconnect(public_url)